### Stacking for Classification

In [39]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

from sklearn.ensemble import StackingClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.neighbors import KNeighborsClassifier
from sklearn.naive_bayes import GaussianNB
from sklearn.ensemble import GradientBoostingClassifier
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

import warnings
warnings.filterwarnings('ignore')

In [40]:
df = pd.read_csv('heart.csv')
df.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal,target
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2,1
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2,1
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2,1
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2,1


In [41]:
X = df.drop('target', axis=1)
y = df['target']

In [42]:
X.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
0,63,1,3,145,233,1,0,150,0,2.3,0,0,1
1,37,1,2,130,250,0,1,187,0,3.5,0,0,2
2,41,0,1,130,204,0,0,172,0,1.4,2,0,2
3,56,1,1,120,236,0,1,178,0,0.8,2,0,2
4,57,0,0,120,354,0,1,163,1,0.6,2,0,2


In [43]:
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=42
)
X_train.shape, X_test.shape, y_train.shape, y_test.shape

((242, 13), (61, 13), (242,), (61,))

In [44]:
scale = StandardScaler()

var_transform = ['thalach', 'age', 'trestbps', 'oldpeak', 'chol']
X_train[var_transform] = scale.fit_transform(X_train[var_transform])
X_test[var_transform] = scale.transform(X_test[var_transform])

In [45]:
X_train.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
132,-1.356798,1,1,-0.616856,0.914034,0,1,0.532781,0,-0.920864,2,0,2
202,0.385086,1,0,1.169491,0.439527,0,0,-1.753582,1,-0.193787,2,0,3
196,-0.921327,1,2,1.169491,-0.300704,0,1,-0.139679,0,2.350982,1,0,2
75,0.058483,0,1,0.276318,0.059921,0,0,0.487950,0,0.351521,1,0,2
176,0.602822,1,0,-0.795490,-0.319684,1,1,0.443119,1,0.351521,2,2,3


First Layer Estimators

In [46]:
base_estimators = [
    ('knn', KNeighborsClassifier()),
    ('nb', GaussianNB()),
    ('gb', GradientBoostingClassifier())
]

In [47]:
clf = StackingClassifier(
    estimators = base_estimators,
    final_estimator = LogisticRegression(),
    cv =10
)

In [48]:
clf.fit(X_train, y_train)

StackingClassifier(cv=10,
                   estimators=[('knn', KNeighborsClassifier()),
                               ('nb', GaussianNB()),
                               ('gb', GradientBoostingClassifier())],
                   final_estimator=LogisticRegression())

In [49]:
yhat = clf.predict(X_test)

In [50]:
accuarace = accuracy_score(y_test, yhat)
print('Accuracy: %.3f' % (accuarace*100))

Accuracy: 90.164


In [51]:
X_test.head()

,age,sex,cp,trestbps,chol,fbs,restecg,thalach,exang,oldpeak,slope,ca,thal
179,0.276218,1,0,1.169491,0.553408,0,0,-1.708752,1,-0.375556,1,1,1
228,0.493954,1,3,2.360389,0.781172,0,0,0.398289,0,-0.739095,1,0,3
111,0.276218,1,2,1.169491,-2.293633,1,1,1.025918,0,-0.739095,2,1,3
246,0.167350,0,0,0.216773,3.077785,0,0,-0.005187,1,0.805944,1,2,3
60,1.800367,0,2,-1.212304,0.344625,1,0,-0.901800,0,-0.920864,2,1,2


In [52]:
y_test.head()

,target
179,0
228,0
111,1
246,0
60,1


In [53]:
clf.predict(np.array([0.276218,	1,	2,	1.169491,	-2.293633,	1,	1,	1.025918,	0,	-0.739095,	2,	1,	3]).reshape(1,-1))

array([1])

Stacking For Regression

In [54]:
from sklearn.ensemble import StackingRegressor
from sklearn.linear_model import LinearRegression
from sklearn.svm import SVR
from sklearn.neighbors import KNeighborsRegressor
from sklearn.tree import DecisionTreeRegressor
from sklearn.model_selection import train_test_split
from sklearn.metrics import mean_squared_error

In [55]:
data_url = "http://lib.stat.cmu.edu/datasets/boston"
raw_df = pd.read_csv(data_url, sep="\s+", skiprows=22, header=None)
data = np.hstack([raw_df.values[::2, :], raw_df.values[1::2, :2]])
target = raw_df.values[1::2, 2]

In [56]:
df = pd.DataFrame(data, columns=['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'RAD', 'TAX', 'PTRATIO', 'B', 'LSTAT'])
df['MEDV'] = target
df.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT,MEDV
0,0.00632,18.0,2.31,0.0,0.538,6.575,65.2,4.0900,1.0,296.0,15.3,396.90,4.98,24.0
1,0.02731,0.0,7.07,0.0,0.469,6.421,78.9,4.9671,2.0,242.0,17.8,396.90,9.14,21.6
2,0.02729,0.0,7.07,0.0,0.469,7.185,61.1,4.9671,2.0,242.0,17.8,392.83,4.03,34.7
3,0.03237,0.0,2.18,0.0,0.458,6.998,45.8,6.0622,3.0,222.0,18.7,394.63,2.94,33.4
4,0.06905,0.0,2.18,0.0,0.458,7.147,54.2,6.0622,3.0,222.0,18.7,396.90,5.33,36.2


In [57]:
X = df.drop('MEDV', axis = 1)
y = df.MEDV

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

X_train.shape, X_test.shape, y_train.shape, y_test.shape

((404, 13), (102, 13), (404,), (102,))

Standardized

In [58]:
scale = StandardScaler()

var_transform = ['CRIM', 'ZN', 'INDUS', 'CHAS', 'NOX', 'RM', 'AGE', 'DIS', 'TAX', 'PTRATIO', 'B', 'LSTAT']
X_train[var_transform] = scale.fit_transform(X_train[var_transform])
X_test[var_transform] = scale.transform(X_test[var_transform])

In [59]:
X_train.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
477,1.287702,-0.500320,1.033237,-0.278089,0.489252,-1.428069,1.028015,-0.802173,24.0,1.578434,0.845343,-0.074337,1.753505
15,-0.336384,-0.500320,-0.413160,-0.278089,-0.157233,-0.680087,-0.431199,0.324349,4.0,-0.584648,1.204741,0.430184,-0.561474
332,-0.403253,1.013271,-0.715218,-0.278089,-1.008723,-0.402063,-1.618599,1.330697,1.0,-0.602724,-0.637176,0.065297,-0.651595
423,0.388230,-0.500320,1.033237,-0.278089,0.489252,-0.300450,0.591681,-0.839240,24.0,1.578434,0.845343,-3.868193,1.525387
19,-0.325282,-0.500320,-0.413160,-0.278089,-0.157233,-0.831094,0.033747,-0.005494,4.0,-0.584648,1.204741,0.379119,-0.165787


Base Estimators

In [60]:
base_estimators = [
    ('knn', KNeighborsRegressor()),
    ('svr', SVR()),
    ('dt', DecisionTreeRegressor())
]

In [61]:
reg = StackingRegressor(
    estimators = base_estimators,
    final_estimator = LinearRegression(),
    cv = 10
)

In [63]:
reg.fit(X_train, y_train)

StackingRegressor(cv=10,
                  estimators=[('knn', KNeighborsRegressor()), ('svr', SVR()),
                              ('dt', DecisionTreeRegressor())],
                  final_estimator=LinearRegression())

In [64]:
yhat = reg.predict(X_test)

In [66]:
mse = mean_squared_error(y_test, yhat)
print(f"Mean Squared Error: {mse}")

Mean Squared Error: 10.693437504610102


In [67]:
X_test.head()

,CRIM,ZN,INDUS,CHAS,NOX,RM,AGE,DIS,RAD,TAX,PTRATIO,B,LSTAT
173,-0.396809,-0.500320,-1.007111,-0.278089,-0.395412,0.141282,0.555916,-0.545853,5.0,-0.650926,-0.771951,0.428872,-0.481210
274,-0.400796,1.229499,-0.664391,3.595975,-0.931315,0.623942,-1.275254,0.126565,4.0,-0.903989,-0.322703,0.444180,-1.257094
491,-0.395234,-0.500320,2.433163,-0.278089,0.446720,-0.469805,1.081663,-0.911447,4.0,1.849573,0.800418,0.369934,0.790338
72,-0.396825,-0.500320,-0.025421,-0.278089,-1.220532,-0.354079,-2.172957,0.694876,4.0,-0.596698,0.396095,0.378682,-0.976875
452,0.167084,-0.500320,1.033237,-0.278089,1.331384,-0.026661,0.831307,-0.676503,24.0,1.578434,0.845343,0.315043,0.677687


In [68]:
y_test.head()

,MEDV
173,23.6
274,32.4
491,13.6
72,22.8
452,16.1


In [70]:
reg.predict(np.array([
    -0.395234,	-0.500320,	2.433163,	-0.278089,	0.446720,	-0.469805,	1.081663,	-0.911447,	4.0,	1.849573,	0.800418,	0.369934,	0.790338,
]).reshape(1,-1))

array([16.94814745])